<a href="https://colab.research.google.com/github/pendulumswings/pendulumswings.github.io/blob/main/scripts/chapter2/relative_power.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Relative Power harmonic contribution for the simple pendulum oscillations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import ellipk

# 1. Define the four test cases
angles_deg = [60.0, 90.0, 140.0, 170.0]

# Define the harmonic numbers (m = 1, 3, 5, 7, 9... up to 21)
m_values = np.arange(1, 22, 2)

# Create a 2x2 subplot grid (sharing the x and y axes so the scale is identical)
fig, axs = plt.subplots(2, 2, figsize=(12, 10), sharey=True, sharex=True)
axs = axs.flatten()

for i, theta_0_deg in enumerate(angles_deg):
    theta_0 = np.radians(theta_0_deg)

    # Calculate the elliptic modulus and complete integrals
    k = np.sin(theta_0 / 2.0)
    K = ellipk(k**2)
    K_prime = ellipk(1.0 - k**2)

    # Calculate the Nome (q)
    q = np.exp(-np.pi * K_prime / K)

    # Calculate exact amplitude (b_m) for each harmonic
    b_m = (8.0 / m_values) * (q**(m_values / 2.0) / (1.0 + q**m_values))

    # Calculate relative amplitude percentages for display
    pct_3w = (b_m[1] / b_m[0]) * 100
    pct_5w = (b_m[2] / b_m[0]) * 100

    # Calculate absolute and relative power
    P_absolute = 0.5 * (b_m**2)
    P_relative = P_absolute / P_absolute[0]
    log_P_relative = np.log10(P_relative)

    # --- Plotting ---
    ax = axs[i]

    # Stem plot for the FFT-style spikes
    markerline, stemlines, baseline = ax.stem(m_values, log_P_relative, basefmt=" ")
    plt.setp(markerline, marker='o', markersize=7, color='#0ea5e9', markeredgecolor='black')
    plt.setp(stemlines, color='#0ea5e9', linewidth=2)

    # Linear trendline showing the mathematical drop-off
    trend_coef = np.polyfit(m_values, log_P_relative, 1)
    trendline = np.poly1d(trend_coef)
    ax.plot(m_values, trendline(m_values), color='red', linestyle='--', alpha=0.5)

    # Add text box with the amplitude contributions
    text_str = f"Nome ($q$) $\\approx$ {q:.4f}\n$3\\omega$ Amp: {pct_3w:.2f}%\n$5\\omega$ Amp: {pct_5w:.3f}%"
    ax.text(0.05, 0.05, text_str, transform=ax.transAxes, fontsize=10,
            verticalalignment='bottom', horizontalalignment='left',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))

    # Axes and Labels
    ax.set_title(f"$\\theta_0 = {theta_0_deg}^\\circ$", fontsize=14, fontweight='bold')
    ax.set_xticks(m_values)
    ax.grid(True, alpha=0.3, linestyle='--')

    # Only set x-labels on the bottom row
    if i >= 2:
        ax.set_xlabel("Harmonic Number $m$", fontsize=12)

    # Only set y-labels on the left column
    if i % 2 == 0:
        ax.set_ylabel("Log$_{10}$ of Relative Power", fontsize=12)

plt.tight_layout()
plt.show()